# Lesson 3.4 — MCP Server Development: Resources, Tools, and Prompts

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain the Model Context Protocol and its client/server roles
- Author an MCP server exposing tools, resources, and prompts
- Describe transports: stdio vs sockets/HTTP
- Run and integrate the server with a Claude application

**You will produce:** A runnable MCP server stub file (via %%writefile) exposing a tool, resource, and prompt.

## Concepts

The **Model Context Protocol (MCP)** is an open standard for connecting LLM applications to external capabilities. An **MCP server** exposes three primitives:

- **Tools**: callable functions (like tool use, but reusable across apps).
- **Resources**: readable data (files, records) addressed by URI.
- **Prompts**: reusable prompt templates the client can invoke.

A **client** (e.g., Claude Desktop, Claude Code, or your app) connects to the server over a **transport**: **stdio** (local subprocess, simplest) or **HTTP/sockets** (networked). Building an MCP server makes a capability reusable and independently maintainable - exactly the scenario in Sample Question 3 (answer: B).

> **Reference:** [Model Context Protocol docs](https://modelcontextprotocol.io/introduction)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Author an MCP server stub

Write a server file using the Python `mcp` SDK (FastMCP). We use `%%writefile` so you get a real file you can run. Install with `pip install mcp`.

In [ ]:
%%writefile inventory_server.py
"""Minimal MCP server exposing a tool, a resource, and a prompt.
Run with: python inventory_server.py  (speaks MCP over stdio)
Docs: https://modelcontextprotocol.io  |  pip install mcp
"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("inventory")

_STOCK = {"widget": 12, "gadget": 0, "gizmo": 5}

@mcp.tool()
def check_stock(item: str) -> int:
    """Return the current stock count for an item."""
    return _STOCK.get(item, 0)

@mcp.resource("inventory://catalog")
def catalog() -> str:
    """Expose the full catalog as a readable resource."""
    return "\n".join(f"{k}: {v}" for k, v in _STOCK.items())

@mcp.prompt()
def restock_prompt(item: str) -> str:
    """A reusable prompt template for drafting a restock request."""
    return f"Draft a restock order for '{item}'. Check stock first, then recommend a quantity."

if __name__ == "__main__":
    mcp.run()  # stdio transport by default

## Code-Along: Step 2 — Verify the file and show how to run it

Confirm the stub exists; the bash cell shows the run command (requires `pip install mcp`).

In [ ]:
import os
print("server written:", os.path.exists("inventory_server.py"))
print("Run it with: python inventory_server.py (speaks MCP over stdio)")

## Code-Along: Step 3 — How a client connects

A Claude application or Claude Desktop registers the server command. Example client config (JSON) a host would use:

In [ ]:
client_config = {
    "mcpServers": {
        "inventory": {
            "command": "python",
            "args": ["inventory_server.py"]
        }
    }
}
import json
print(json.dumps(client_config, indent=2))

## Exercise

Extend `inventory_server.py` with a second tool `set_stock(item, qty)` that updates the dict and returns the new count. Rewrite the file with `%%writefile` and verify it exists.

In [ ]:
# TODO: %%writefile inventory_server.py  (copy the server and add set_stock)
# TODO: @mcp.tool() def set_stock(item: str, qty: int) -> int: ...
# TODO: verify with os.path.exists('inventory_server.py')

## Key Takeaways

- MCP standardizes connecting Claude apps to external tools, resources, and prompts.
- An MCP server exposes tools (actions), resources (data by URI), and prompts (templates).
- stdio is the simplest transport; HTTP/sockets enable networked servers.
- Build an MCP server when a capability must be reusable across apps (Sample Q3: B).

---

**Next:** [Lesson 3.5 — Choosing Among Built-in Tools, Custom Tools, Skills, and MCPs](./lesson-3.5-choosing-tool-approach.ipynb)